# 1. Librairies importation

In [5]:
%pip install -r ../requirements.txt

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import GroupKFold

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import sys

# Works when running from the project root or notebooks/
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import canonicalize_smiles

In [8]:
from tdc.single_pred import ADME, Tox
import pandas as pd

# Save datasets in the project-level data directory
raw_dir = project_root / "data"
raw_dir.mkdir(parents=True, exist_ok=True)

# Load regression dataset
solubility = ADME(name="Solubility_AqSolDB", path=str(raw_dir))
df_reg = solubility.get_data()
df_reg.to_csv(raw_dir / "solubility_aqsoldb.tab", sep="\t", index=False)

# Preview datasets and report shapes
display(df_reg.head(10))
print("Regression:", df_reg.shape)

Found local copy...
Loading...
Done!


,Drug_ID,Drug,Y
0,"N,N,N-trimethyloctadecan-1-aminium bromide",CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],-3.616127
1,Benzo[cd]indol-2(1H)-one,O=C1Nc2cccc3cccc1c23,-3.254767
2,4-chlorobenzaldehyde,O=Cc1ccc(Cl)cc1,-2.177078
3,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,-3.924409
4,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,-4.662065
5,vinyltoluene,C=Cc1cccc(C)c1,-3.123150
6,3-(3-ethylcyclopentyl)propanoic acid,CCC1CCC(CCC(=O)O)C1,-3.286116
7,"11,16,17,21-tetrahydroxypregna-1,4-diene-3,20-...",CC12C=CC(=O)C=C1CCC1C2C(O)CC2(C)C1CC(O)C2(O)C(...,-2.664549
8,bis(4-fluorophenyl)methanone,O=C(c1ccc(F)cc1)c1ccc(F)cc1,-4.396652
9,1-[2-(benzoyloxy)propoxy]propan-2-yl benzoate ...,O=C(OCCCOCCCOC(=O)c1ccccc1)c1ccccc1,-4.595503


Regression: (9982, 3)


# 2. Adapating our columns


In [13]:
print(df_reg.columns.tolist())

smiles_col = "drug"

['Drug_ID', 'Drug', 'Y']


# 3. Bemis-Murcko scaffolds to separate molecules with the same scaffold in training and test

In [14]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    return Chem.MolToSmiles(scaffold, canonical=True)


df_reg["scaffold"] = df_reg[smiles_col].apply(get_scaffold)


invalid_count = df_reg["scaffold"].isna().sum()
print("Invalid SMILES:", invalid_count)

# Remove molecules with invalid structures
df_reg = df_reg.dropna(subset=["scaffold"]).copy()

print("Valid molecules:", len(df_reg))
print("Unique scaffolds:", df_reg["scaffold"].nunique())

KeyError: 'drug'

# 4. Test-split 

In [ ]:
# Size of each scaffold group
scaffold_sizes = df_reg["scaffold"].value_counts()

print(scaffold_sizes.describe())
print("Largest scaffold group:", scaffold_sizes.max())

# Splitting our molecules by scafold 
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df_reg, groups=df_reg["scaffold"])
)

train_df = df_reg.iloc[train_idx].copy()
test_df = df_reg.iloc[test_idx].copy()

print("Training set:", len(train_df), "molecules")
print("Test set:", len(test_df), "molecules")
print("Test proportion:", round(len(test_df) / len(df_reg) * 100, 2), "%")


# 5. Freeze the split from test-split 

In [ ]:
df_reg["split"] = "train"
df_reg.loc[test_df.index, "split"] = "test"

df_reg.to_csv("esol_scaffold_split.csv", index=False)

print(df_reg["split"].value_counts())
print("Split assignments saved.")